# Parser Inference FastAPI + ngrok on 

In [1]:
!pip install -q fastapi "uvicorn[standard]" pyngrok transformers peft accelerate bitsandbytes safetensors pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import json
import time
import zipfile
import tempfile
from pathlib import Path
from typing import Any
import threading

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import uvicorn
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [16]:
BASE_MODEL_ID = os.getenv("PARSER_BASE_MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507")

ARTIFACT_ZIP_PATH = os.getenv(
    "PARSER_ARTIFACT_ZIP",
    "/kaggle/input/government-parser-qwen3-4b-lora-artifact/government_parser_qwen3_4b_lora_artifact.zip",
)

ADAPTER_PATH = os.getenv(
    "PARSER_ADAPTER_PATH",
    "/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter"
)

EXTRACT_DIR = "/kaggle/working/government_parser_artifact"
PORT = int(os.getenv("PARSER_SERVICE_PORT", "8000"))

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_AUTHTOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")

MAX_NEW_TOKENS = int(os.getenv("PARSER_MAX_NEW_TOKENS", "512"))
TEMPERATURE = float(os.getenv("PARSER_TEMPERATURE", "0.0"))
TOP_P = float(os.getenv("PARSER_TOP_P", "0.95"))
LOAD_IN_4BIT = os.getenv("PARSER_LOAD_IN_4BIT", "true").lower() == "true"

SERVICE_VERSION = "parser_model_service_v0.1.0"

config_summary = {
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "ARTIFACT_ZIP_PATH": ARTIFACT_ZIP_PATH,
    "ADAPTER_PATH": ADAPTER_PATH,
    "EXTRACT_DIR": EXTRACT_DIR,
    "PORT": PORT,
    "NGROK_AUTHTOKEN_SET": bool(NGROK_AUTHTOKEN),
    "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
    "TEMPERATURE": TEMPERATURE,
    "TOP_P": TOP_P,
    "LOAD_IN_4BIT": LOAD_IN_4BIT,
    "SERVICE_VERSION": SERVICE_VERSION,
}

print(json.dumps(config_summary, indent=2, ensure_ascii=False))

{
  "BASE_MODEL_ID": "Qwen/Qwen3-4B-Instruct-2507",
  "ARTIFACT_ZIP_PATH": "/kaggle/input/government-parser-qwen3-4b-lora-artifact/government_parser_qwen3_4b_lora_artifact.zip",
  "ADAPTER_PATH": "/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter",
  "EXTRACT_DIR": "/kaggle/working/government_parser_artifact",
  "PORT": 8000,
  "NGROK_AUTHTOKEN_SET": true,
  "MAX_NEW_TOKENS": 512,
  "TEMPERATURE": 0.0,
  "TOP_P": 0.95,
  "LOAD_IN_4BIT": true,
  "SERVICE_VERSION": "parser_model_service_v0.1.0"
}


In [4]:
def _find_adapter_dir(root: Path) -> Path | None:
    direct_candidates = [root, root / "final_adapter"]
    for candidate in direct_candidates:
        if candidate.exists() and (candidate / "adapter_config.json").exists():
            return candidate

    for config_path in root.rglob("adapter_config.json"):
        return config_path.parent

    return None


def prepare_adapter_path() -> str:
    if ADAPTER_PATH:
        adapter_path = Path(ADAPTER_PATH).expanduser().resolve()
        if adapter_path.exists() and (adapter_path / "adapter_config.json").exists():
            print(f"Using adapter folder from PARSER_ADAPTER_PATH: {adapter_path}")
            return str(adapter_path)
        raise FileNotFoundError(
            f"PARSER_ADAPTER_PATH is set but adapter_config.json was not found: {adapter_path}"
        )

    artifact_zip_path = Path(ARTIFACT_ZIP_PATH).expanduser()
    if artifact_zip_path.exists():
        extract_dir = Path(EXTRACT_DIR)
        extract_dir.mkdir(parents=True, exist_ok=True)

        print(f"Extracting adapter artifact zip: {artifact_zip_path}")
        with zipfile.ZipFile(artifact_zip_path, "r") as artifact_zip:
            artifact_zip.extractall(extract_dir)

        adapter_dir = _find_adapter_dir(extract_dir)
        if adapter_dir is not None:
            print(f"Using extracted adapter folder: {adapter_dir}")
            return str(adapter_dir)

        raise FileNotFoundError(
            f"Artifact zip was extracted to {extract_dir}, but no folder containing adapter_config.json was found. "
            "Do not use paths like artifact.zip\\final_adapter on Kaggle; extract the zip first or set PARSER_ADAPTER_PATH."
        )

    raise FileNotFoundError(
        "Could not find the parser LoRA adapter. Set PARSER_ADAPTER_PATH to an extracted final_adapter folder, "
        f"or set PARSER_ARTIFACT_ZIP to a real zip file. Current zip path: {ARTIFACT_ZIP_PATH}"
    )

In [5]:
REQUIRED_FIELDS = [
    "intent",
    "question_family",
    "indicators",
    "countries",
    "country_groups",
    "start_year",
    "end_year",
    "relative_time",
    "event_time",
    "ranking_order",
    "limit",
    "threshold",
    "aggregation",
    "chart_preference",
    "needs_clarification",
    "clarification_questions",
    "confidence",
]

DEFAULT_FALLBACK_PARSED = {
    "intent": "NEED_CLARIFICATION",
    "question_family": "parser_error",
    "indicators": [],
    "countries": [],
    "country_groups": [],
    "start_year": None,
    "end_year": None,
    "relative_time": None,
    "event_time": None,
    "ranking_order": None,
    "limit": None,
    "threshold": None,
    "aggregation": None,
    "chart_preference": "none",
    "needs_clarification": True,
    "clarification_questions": [
        "Mình chưa phân tích chắc chắn được câu hỏi. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
    ],
    "confidence": 0.0,
}

In [6]:
def extract_first_json_object(text: str) -> str | None:
    if not text:
        return None

    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()

    start = cleaned.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escaped = False

    for index in range(start, len(cleaned)):
        char = cleaned[index]

        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return cleaned[start : index + 1]

    return None


def parse_model_json(raw_text: str) -> tuple[dict | None, bool, str | None]:
    json_text = extract_first_json_object(raw_text)
    if json_text is None:
        return None, False, "no_json_object_found"

    try:
        parsed = json.loads(json_text)
    except json.JSONDecodeError as exc:
        return None, False, str(exc)

    if not isinstance(parsed, dict):
        return None, False, "json_value_is_not_object"

    return parsed, True, None


def basic_schema_check(parsed: dict) -> tuple[bool, list[str]]:
    errors: list[str] = []

    missing_fields = [field for field in REQUIRED_FIELDS if field not in parsed]
    if missing_fields:
        errors.append(f"missing_required_fields: {missing_fields}")

    if "intent" in parsed and not isinstance(parsed["intent"], str):
        errors.append("intent_must_be_str")
    if "question_family" in parsed and not isinstance(parsed["question_family"], str):
        errors.append("question_family_must_be_str")

    for field in ["indicators", "countries", "country_groups", "clarification_questions"]:
        if field in parsed and not isinstance(parsed[field], list):
            errors.append(f"{field}_must_be_list")

    if "needs_clarification" in parsed and not isinstance(parsed["needs_clarification"], bool):
        errors.append("needs_clarification_must_be_bool")

    if "confidence" in parsed and parsed["confidence"] is not None and not isinstance(parsed["confidence"], (int, float)):
        errors.append("confidence_must_be_number_or_null")

    return len(errors) == 0, errors

In [7]:
def build_parser_messages(message: str, context: dict | None = None) -> list[dict]:
    system_prompt = """You are a semantic parser for a Government Economic Indicator AI Agent.
Return only one valid JSON object matching ParsedQuery v1.
Do not answer the user question.
Do not write SQL.
Do not query data.
Do not invent missing indicators, countries, years, or question families.
If required information is missing, return NEED_CLARIFICATION.
Prefer NEED_CLARIFICATION over guessing.

Required JSON fields:
intent, question_family, indicators, countries, country_groups, start_year, end_year,
relative_time, event_time, ranking_order, limit, threshold, aggregation,
chart_preference, needs_clarification, clarification_questions, confidence.

Use null for unknown scalar fields.
Use [] for unknown list fields.
Use "none" for chart_preference if no chart is requested.
Output JSON only."""

    user_content = {
        "message": message,
        "context": context,
        "instruction": "Output JSON only.",
    }

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(user_content, ensure_ascii=False, indent=2)},
    ]

In [8]:
tokenizer = None
model = None
adapter_real_path = None
load_error = None
model_loaded = False
tokenizer_loaded = False
adapter_loaded = False


def load_parser_model():
    global tokenizer, model, adapter_real_path, load_error
    global model_loaded, tokenizer_loaded, adapter_loaded

    load_error = None
    model_loaded = False
    tokenizer_loaded = False
    adapter_loaded = False

    try:
        adapter_real_path = prepare_adapter_path()

        try:
            tokenizer = AutoTokenizer.from_pretrained(adapter_real_path, trust_remote_code=True)
            print(f"Loaded tokenizer from adapter path: {adapter_real_path}")
        except Exception as tokenizer_error:
            print(f"Could not load tokenizer from adapter path: {tokenizer_error}")
            print(f"Falling back to base tokenizer: {BASE_MODEL_ID}")
            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer_loaded = True

        quantization_config = None
        if LOAD_IN_4BIT and torch.cuda.is_available():
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
                bnb_4bit_use_double_quant=True,
            )
        elif LOAD_IN_4BIT:
            print("4-bit loading was requested, but CUDA is not available. Loading without 4-bit quantization.")

        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            device_map="auto",
            quantization_config=quantization_config,
            trust_remote_code=True,
        )

        model = PeftModel.from_pretrained(base_model, adapter_real_path)
        model.eval()

        adapter_loaded = True
        model_loaded = True

        print("Parser model loaded successfully.")
        return model
    except Exception as exc:
        load_error = f"{type(exc).__name__}: {exc}"
        print(f"Parser model load failed: {load_error}")
        raise


try:
    load_parser_model()
except Exception:
    print("Continuing with degraded service state. /health will expose load_error and /parse will return HTTP 503 until the model loads successfully.")

Using adapter folder from PARSER_ADAPTER_PATH: /kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter


Loaded tokenizer from adapter path: /kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Parser model loaded successfully.


In [9]:
def generate_parser_output(message: str, context: dict | None = None) -> str:
    if model is None or tokenizer is None:
        raise RuntimeError(f"Parser model is not loaded. load_error={load_error}")

    messages = build_parser_messages(message, context)

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    else:
        prompt = "\n\n".join(f"{item['role'].upper()}: {item['content']}" for item in messages)
        prompt = f"{prompt}\n\nASSISTANT:"
        inputs = tokenizer(prompt, return_tensors="pt")

    target_device = next(model.parameters()).device
    inputs = {key: value.to(target_device) for key, value in inputs.items()}

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": TEMPERATURE > 0,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if TEMPERATURE > 0:
        generation_kwargs["temperature"] = TEMPERATURE
        generation_kwargs["top_p"] = TOP_P

    with torch.inference_mode():
        output_ids = model.generate(**inputs, **generation_kwargs)

    input_length = inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0][input_length:]
    if generated_ids.numel() == 0:
        generated_ids = output_ids[0]

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [10]:
class ParseRequest(BaseModel):
    message: str
    context: dict[str, Any] | None = None
    debug: bool = False


class ParseResponse(BaseModel):
    parsed: dict[str, Any]
    raw_model_output: str | None = None
    valid_json: bool
    schema_pass: bool
    catalog_pass: bool
    safe_to_execute: bool
    fallback_reason: str | None = None
    repairs_applied: list[str] = Field(default_factory=list)
    candidates: dict[str, Any] = Field(default_factory=dict)
    latency_ms: int

In [11]:
app = FastAPI(title="Government Parser Model Service", version=SERVICE_VERSION)


@app.get("/health")
def health() -> dict[str, Any]:
    if model_loaded and tokenizer_loaded and adapter_loaded:
        status = "ok"
    elif load_error:
        status = "error"
    else:
        status = "degraded"

    return {
        "status": status,
        "service": "parser-model-service",
        "version": SERVICE_VERSION,
        "model_loaded": model_loaded,
        "tokenizer_loaded": tokenizer_loaded,
        "adapter_loaded": adapter_loaded,
        "base_model_id": BASE_MODEL_ID,
        "adapter_path": adapter_real_path,
        "load_error": load_error,
    }


@app.post("/parse", response_model=ParseResponse)
def parse(request: ParseRequest) -> ParseResponse:
    start_time = time.time()
    message = request.message.strip()

    if not message:
        raise HTTPException(status_code=400, detail="message must not be empty")

    if not (model_loaded and tokenizer_loaded and adapter_loaded):
        raise HTTPException(
            status_code=503,
            detail={"message": "parser model is not loaded", "load_error": load_error},
        )

    raw = generate_parser_output(message, request.context)
    parsed, valid_json, json_error = parse_model_json(raw)

    fallback_reason = None
    schema_pass = False
    repairs_applied: list[str] = []

    if not valid_json or parsed is None:
        parsed = DEFAULT_FALLBACK_PARSED.copy()
        fallback_reason = "invalid_json"
        repairs_applied.append(f"fallback_due_to_json_error: {json_error}")
    else:
        schema_pass, schema_errors = basic_schema_check(parsed)
        if schema_pass:
            fallback_reason = "catalog_validator_not_implemented_phase2"
        else:
            fallback_reason = "schema_error"
            repairs_applied.extend(schema_errors)

    latency_ms = int((time.time() - start_time) * 1000)

    return ParseResponse(
        parsed=parsed,
        raw_model_output=raw if request.debug else None,
        valid_json=valid_json,
        schema_pass=schema_pass,
        catalog_pass=False,
        safe_to_execute=False,
        fallback_reason=fallback_reason,
        repairs_applied=repairs_applied,
        candidates={},
        latency_ms=latency_ms,
    )

In [12]:
test_payload = {
    "message": "So sánh nợ công Việt Nam và Thái Lan từ 2010 đến 2023",
    "context": None,
    "debug": True,
}

if model_loaded and tokenizer_loaded and adapter_loaded:
    raw_test_output = generate_parser_output(test_payload["message"], test_payload["context"])
    parsed_test_output, valid_test_json, test_json_error = parse_model_json(raw_test_output)
    schema_test_pass, schema_test_errors = basic_schema_check(parsed_test_output) if parsed_test_output else (False, [])

    print("Raw model output:")
    print(raw_test_output)
    print("\nParsed result:")
    print(json.dumps(parsed_test_output, indent=2, ensure_ascii=False))
    print("\nValidation:")
    print(json.dumps({
        "valid_json": valid_test_json,
        "json_error": test_json_error,
        "schema_pass": schema_test_pass,
        "schema_errors": schema_test_errors,
    }, indent=2, ensure_ascii=False))
else:
    print(f"Skipping local smoke test because model is not loaded. load_error={load_error}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw model output:
{"intent":"COMPARE_COUNTRIES","question_family":"compare_countries_volatility","indicators":["govdebt_GDP"],"countries":["VNM","THA"],"country_groups":[],"start_year":2010,"end_year":2023,"relative_time":null,"event_time":null,"ranking_order":null,"limit":null,"threshold":null,"aggregation":null,"chart_preference":"none","needs_clarification":false,"clarification_questions":[],"confidence":1.0}

Parsed result:
{
  "intent": "COMPARE_COUNTRIES",
  "question_family": "compare_countries_volatility",
  "indicators": [
    "govdebt_GDP"
  ],
  "countries": [
    "VNM",
    "THA"
  ],
  "country_groups": [],
  "start_year": 2010,
  "end_year": 2023,
  "relative_time": null,
  "event_time": null,
  "ranking_order": null,
  "limit": null,
  "threshold": null,
  "aggregation": null,
  "chart_preference": "none",
  "needs_clarification": false,
  "clarification_questions": [],
  "confidence": 1.0
}

Validation:
{
  "valid_json": true,
  "json_error": null,
  "schema_pass": true

In [18]:
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT)


server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print(f"FastAPI server running on port {PORT}")

FastAPI server running on port 8000


INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [ ]:
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
else:
    print("WARNING: NGROK_AUTHTOKEN is not set. Set os.environ['NGROK_AUTHTOKEN'] or configure it as a Kaggle Secret before starting the tunnel.")

public_url = ngrok.connect(PORT, "http")
print("Public URL:", public_url)
print(f"GET  {public_url}/health")
print(f"POST {public_url}/parse")